# Quantum Teleportation

## **Quantum teleportation** is a protocol for transmitting an unknown quantum state $|\psi\rangle$ from Alice to Bob using one Bell pair and two classical bits. No actual qubit is sent — only classical information — but the state appears at Bob's location.

# 1. The setup

## - Alice has an unknown qubit $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ she wants to send to Bob.
## - Alice and Bob each hold one half of a pre-shared Bell pair $|\Phi^+\rangle$.
## - Alice can send two classical bits to Bob over a normal phone line.

## Three qubits total: $q_0$ = Alice's unknown, $q_1$ = Alice's half of the Bell pair, $q_2$ = Bob's half.

# 2. The protocol

## 1. Alice applies CNOT with $q_0$ as control and $q_1$ as target.
## 2. Alice applies $H$ to $q_0$.
## 3. Alice measures $q_0$ and $q_1$ in the computational basis, getting two classical bits $(a, b)$.
## 4. Alice sends $(a, b)$ to Bob (this uses the classical channel).
## 5. Bob applies $X^b Z^a$ to his qubit $q_2$.

## After step 5, Bob's qubit is in state $|\psi\rangle$. Alice's original qubit has collapsed and no longer holds $|\psi\rangle$ — **no-cloning is respected** (chapter 1 notebook 7).

# 3. Why it works (algebra)

## After steps 1 and 2 (before measurement), expand the three-qubit state in the computational basis of the first two qubits:

$$ \Large  \tfrac{1}{2}\Big[ |00\rangle(\alpha|0\rangle + \beta|1\rangle) + |01\rangle(\alpha|1\rangle + \beta|0\rangle) + |10\rangle(\alpha|0\rangle - \beta|1\rangle) + |11\rangle(\alpha|1\rangle - \beta|0\rangle) \Big]. $$

## Each of the four measurement outcomes leaves Bob's qubit in one of four states related to $|\psi\rangle$ by a Pauli:

## - $(0,0)$ → $|\psi\rangle$ → apply $I$
## - $(0,1)$ → $X|\psi\rangle$ → apply $X$
## - $(1,0)$ → $Z|\psi\rangle$ → apply $Z$
## - $(1,1)$ → $XZ|\psi\rangle$ → apply $XZ$

## Hence $X^b Z^a$ on Bob's qubit always recovers $|\psi\rangle$. The classical bits tell him which correction to apply.

In [ ]:
import numpy as np

# Define gates
H = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
CNOT_01 = np.kron(np.array([[1,0],[0,0]]), np.eye(2)) + np.kron(np.array([[0,0],[0,1]]), X)

# Build the 3-qubit operators (qubit 0 = unknown, qubit 1 = Alice's Bell, qubit 2 = Bob's Bell)
I8 = np.eye(8, dtype=complex)
CNOT_01_3 = np.kron(CNOT_01, I)
H_0_3     = np.kron(np.kron(H, I), I)

def teleport(alpha, beta):
    psi = np.array([alpha, beta], dtype=complex)
    # Prepare Bell pair on qubits 1,2
    bell = np.array([1,0,0,1], dtype=complex)/np.sqrt(2)
    state = np.kron(psi, bell)            # 8-dim vector
    # Alice's gates
    state = CNOT_01_3 @ state
    state = H_0_3 @ state
    # Measure qubits 0 and 1; group amplitudes by the 4 outcomes
    out = state.reshape(2, 2, 2)          # axes: q0, q1, q2
    corrections = {(0,0): I, (0,1): X, (1,0): Z, (1,1): X @ Z}
    for (a, b), U in corrections.items():
        bob = out[a, b]                   # Bob's amplitude conditional on (a, b)
        bob_norm = bob / np.linalg.norm(bob)
        recovered = U @ bob_norm
        print(f'a={a}, b={b}: Bob recovers {recovered.round(4)}')
    print('Original |psi>          :', psi.round(4))

teleport(np.cos(0.7), np.sin(0.7)*np.exp(1j*0.3))

# 4. What teleportation does and does not do

## - It **does** transfer an unknown state from one location to another using shared entanglement and classical bits.
## - It **does not** transfer information faster than light — Bob's qubit is in a *random* state until the classical bits arrive.
## - It **does not** clone the state — Alice's qubit is destroyed by the measurement.

## Teleportation is also a key primitive in fault-tolerant quantum computing: many fault-tolerant gates are implemented *by teleportation through a special resource state*.

# 5. Where teleportation shows up

## - **Quantum networks / quantum internet**: long-distance state transfer between nodes.
## - **Measurement-based quantum computation**: the entire model is teleportation-driven.
## - **Fault-tolerant gates**: magic-state injection teleports a magic state into the data register.

# Recap

## - 1 unknown qubit + 1 Bell pair + 2 classical bits = Bob receives $|\psi\rangle$.
## - Protocol: CNOT, $H$, measure $(a,b)$, send classically, apply $X^b Z^a$.
## - No faster-than-light signalling; no cloning.
## - A primitive in quantum networks and fault-tolerant computing.

## Next: the *dual* protocol — sending two classical bits using one qubit.